# Evaluating a MEDS run against the Ithaca test bed

A worked pass over one MEDS run, from the sub-daily biophysics up to the demographic trajectory,
using each output tier for the thing that tier exists to answer:

| tier | window | what it is for here |
|---|---|---|
| **FAST** | one sub-step | the **diurnal** energy and carbon cycles |
| **DAILY** | one day | the **seasonal** cycles |
| **MONTHLY / ANNUAL** | month / year | the **demographic** trajectory, and the closure identities |

**This is not a benchmark.** MEDS v0.2 has never been scored against flux-tower or inventory data,
and nothing below does that. What it does is check the run against things that are knowable without
observations: conservation identities the output must satisfy, and magnitudes and shapes that are
either physically possible or are not. That is the honest ceiling on a pre-benchmark model, and it
is still enough to catch a great deal — every section below has caught something real at least once.

## Producing the run this reads

Any run works, as long as the tiers and groups it uses are enabled. The run this was written
against is three years restarted from a spun-up Ithaca stand, with all four tiers on and
`structure`, `carbon_fluxes`, `water_fluxes` and `energy_fluxes` all `true` in `[output]`:

```toml
[output]
enabled = true
structure = true ; carbon_fluxes = true ; water_fluxes = true ; energy_fluxes = true
[output.fast]
enabled = true
[output.daily]
enabled = true
[output.monthly]
enabled = true
[output.annual]
enabled = true
```

Set `MEDS_RUN` to the output directory, or edit `RUN` in the next cell.

Requires numpy, matplotlib and netCDF4 — the `meds` environment.

In [ ]:
import os, glob, datetime as dt
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset

#----- Point this at an output directory ([output].dir of the run). --------------------------#
RUN = os.environ.get("MEDS_RUN", "runs/ithaca_ark30/out")

def tier_files(tier):
    """Sorted files of one output tier: F fast, D daily, M monthly, Y annual."""
    f = sorted(glob.glob(os.path.join(RUN, f"*-{tier}-*.nc")))
    if not f and tier == "Y":                       # the annual tier is one file, no stamp
        f = sorted(glob.glob(os.path.join(RUN, "*-Y.nc")))
    return f

def read(files, names):
    """Concatenate the named variables over a tier's files, along the record axis.

    Returns a dict of masked arrays. A name the run did not enable comes back as None rather than
    raising -- which variables exist depends on the config, so a reader must cope with absence.
    """
    out = {n: [] for n in names}
    for p in files:
        with Dataset(p) as d:
            for n in names:
                out[n].append(d.variables[n][:] if n in d.variables else None)
    res = {}
    for n, parts in out.items():
        res[n] = None if any(p is None for p in parts) else np.ma.concatenate(parts, axis=0)
    return res

def units(files, name):
    with Dataset(files[0]) as d:
        return getattr(d.variables[name], "units", "") if name in d.variables else ""

F, D, M, Y = (tier_files(t) for t in "FDMY")
print(f"run: {RUN}")
print(f"  FAST    {len(F):5d} files")
print(f"  DAILY   {len(D):5d} files")
print(f"  MONTHLY {len(M):5d} files")
print(f"  ANNUAL  {len(Y):5d} files")
assert D and M, "this notebook needs at least the daily and monthly tiers"

## 1. Does the run conserve?

Three identities that must hold in the **output file**, independently of anything the model asserts
internally. That independence is the point: MEDS's own ledgers close by construction against
quantities it computed, whereas these are read back from netCDF and recombined by a different piece
of code, so a unit error, a wrong aggregation weight or a mis-registered source id shows up here and
nowhere else.

One of them, `Δsoil carbon = litter − Rh`, was silently impossible until v0.2: the litterfall
variables were emitting a per-**second** rate under a per-**year** label, so the identity failed by
a factor of 3.16e7. It was found by checking it.

In [ ]:
#----- ENERGY. Over a long enough window the ground heat flux averages toward zero, so the        #
#      surface balance Rnet = H + LE + G collapses to Rnet ~ H + LE. The daily tier is where this  #
#      is checkable: the fast tier still carries the diurnal storage term. ----------------------#
d = read(D, ["rnet_site", "h_site", "le_site", "resid_energy_site", "resid_water_site"])
rn, h, le = (np.asarray(d[k]).ravel() for k in ("rnet_site", "h_site", "le_site"))
print("ENERGY, daily means over the run [W/m2]")
print(f"   Rnet {rn.mean():8.2f}   H {h.mean():8.2f}   LE {le.mean():8.2f}"
      f"   Rnet-(H+LE) {(rn-h-le).mean():8.2f}")
print(f"   the leftover is the ground+storage term; |mean| / |Rnet| = {abs((rn-h-le).mean())/max(abs(rn.mean()),1e-9):6.3f}")

#----- ...and the model's own whole-column residuals, which should be machine noise. ------------#
for k, u in (("resid_energy_site", "W/m2"), ("resid_water_site", "kg/m2/s")):
    if d[k] is not None:
        v = np.asarray(d[k]).ravel()
        print(f"   {k:<20} mean {v.mean():+10.3e} {u}   worst |.| {np.abs(v).max():.3e}")

### Reading a rate out of a windowed file, without getting it wrong

Every flux variable here is a **rate** averaged over its output window, and every state variable is
a **mean** over that window. Turning a rate back into an amount needs the width of *its own* window:

    amount in window i  =  rate[i] x (t[i+1] - t[i])          # `time` is the period START

Two traps, both of which this notebook hit while being written:

1. **Pairing `rate[i]` with `t[i] - t[i-1]`** — the previous window's width. Harmless while the
   windows are uniform, and badly wrong at the end of a run, where the last window is a partial
   month. Here that turned one disturbance event into a 10.8 kgC/m2 phantom.
2. **Not dropping the trailing partial window.** Its width is unknown (there is no `t[n]`), and a
   once-a-year event landing in a one-day window reports an enormous annualized rate — correctly,
   but you cannot integrate it without the width.

So: use windows `0 .. n-2`, and difference the state over the same span.

In [ ]:
def windows(t):
    """Window widths and the slices that pair fluxes with state, dropping the trailing partial one.

    Returns (flux_slice, widths, lo, hi): amount = (rate[flux_slice] * widths).sum(), and the
    matching state change is state[hi] - state[lo].
    """
    w = np.diff(t)
    lo, hi = 0, len(t) - 2                  # drop the final, partial window
    return slice(lo + 1, hi + 1), w[lo:hi], lo, hi

m = read(M, ["time", "agb_site", "veg_carbon_site", "agb_growth_site", "agb_mort_site",
             "soilc_total_site", "rh_site", "litter_leaf_site", "litter_fineroot_site",
             "litter_struct_site", "mort_carbon_background_site", "mort_carbon_cull_site",
             "mort_carbon_disturb_site"])
t = np.asarray(m["time"]).ravel()
SL, W, LO, HI = windows(t)
amount = lambda name: (np.asarray(m[name]).ravel()[SL] * W).sum()
state  = lambda name: (np.asarray(m[name]).ravel()[HI] - np.asarray(m[name]).ravel()[LO])
print(f"integrating windows {LO+1}..{HI} of {len(t)}   span {t[HI]-t[LO]:.2f} yr")

### The live pool

`d(AGB)` against growth minus mortality. `agb_mort_site` is the **background hazard only** — it is
`mortality rate x AGB` off the cohort diagnostics — so a stand that also loses canopy to treefall
will not close against it, and the size of the miss is the size of the disturbance term.

The mortality-by-pathway variables are **whole-plant** carbon (leaf, fine root, wood and storage),
not AGB, so converting one to the other needs the stand's aboveground fraction. That makes this
identity approximate by construction; the soil one below is not.

In [ ]:
d_agb   = state("agb_site")
growth  = amount("agb_growth_site")
mort_bg = amount("agb_mort_site")
print("LIVE POOL   d(AGB) vs growth - mortality   [kgC/m2 over the span]")
print(f"   growth              {growth:+9.4f}")
print(f"   background mortality{-mort_bg:+9.4f}")
print(f"   --> expected        {growth-mort_bg:+9.4f}      actual d(AGB) {d_agb:+9.4f}"
      f"      gap {d_agb-(growth-mort_bg):+8.4f}  ({abs(d_agb-(growth-mort_bg))/abs(growth-mort_bg):.1%})")

if m["mort_carbon_disturb_site"] is not None:
    agb_frac = (np.asarray(m["agb_site"]).ravel()[HI]
                / np.asarray(m["veg_carbon_site"]).ravel()[HI])
    dist_agb = amount("mort_carbon_disturb_site") * agb_frac
    cull_agb = amount("mort_carbon_cull_site") * agb_frac
    full = growth - mort_bg - dist_agb - cull_agb
    print(f"\n   disturbance mortality, whole plant  {amount('mort_carbon_disturb_site'):+9.4f}")
    print(f"   aboveground fraction of plant carbon {agb_frac:8.3f}  -> AGB lost {-dist_agb:+9.4f}")
    print(f"   --> expected        {full:+9.4f}      actual d(AGB) {d_agb:+9.4f}"
          f"      gap {d_agb-full:+8.4f}  ({abs(d_agb-full)/abs(full):.1%})")
    print("\n   What is left is recruitment, which adds biomass that agb_growth does not carry,\n"
          "   plus the aboveground-fraction approximation above.")

### The soil pool — and a thing the litter variables do not tell you

`d(soil C)` against litterfall minus heterotrophic respiration. This one **should** be exact: every
kilogram entering the CENTURY pools is necromass, and every kilogram leaving is Rh.

It is not — it misses by about 29% — and the reason is worth knowing if you ever read
`litter_*_site`:

> **The litter variables do not carry all the carbon entering the soil.** They are read from the
> per-patch litter accumulator that turnover and background mortality scatter into. The other two
> mortality pathways — a cohort culled at the tracking floor, and the canopy killed when treefall
> opens a gap — add their necromass **directly** to the patch soil-carbon pools, bypassing that
> accumulator entirely.

So closing this budget needs the mortality-by-pathway variables. Before v0.2 added them the
identity was not closable from file at all.

In [ ]:
litter = sum(amount(k) for k in ("litter_leaf_site", "litter_fineroot_site", "litter_struct_site"))
rh     = np.asarray(m["rh_site"]).ravel()[SL].sum()     # already an amount per window, not a rate
d_soil = state("soilc_total_site")

print("SOIL POOL   d(soil C) vs litterfall - Rh   [kgC/m2 over the span]")
print(f"   litterfall (the litter_in accumulator) {litter:+9.4f}")
print(f"   heterotrophic respiration              {-rh:+9.4f}")
naive = litter - rh
print(f"   --> expected {naive:+9.4f}   actual {d_soil:+9.4f}   gap {d_soil-naive:+8.4f}"
      f"  ({abs(d_soil-naive)/abs(naive):.1%})")

if m["mort_carbon_cull_site"] is not None:
    cull, dist = amount("mort_carbon_cull_site"), amount("mort_carbon_disturb_site")
    print(f"\n   ...the two pathways that bypass the accumulator:")
    print(f"   cull (cohorts below the size floor)    {cull:+9.4f}")
    print(f"   canopy killed by treefall disturbance  {dist:+9.4f}")
    full = naive + cull + dist
    print(f"   --> expected {full:+9.4f}   actual {d_soil:+9.4f}   gap {d_soil-full:+8.4f}"
          f"  ({abs(d_soil-full)/abs(full):.2%})")
    print("\n   That is the budget closing. The residual is the window-mean sampling of the state\n"
          "   variables -- soilc_total_site is a MEAN over its window, not an instantaneous value,\n"
          "   so differencing two of them straddles half a window at each end.")

In [ ]:
#----- One magnitude worth checking against something outside the model. A temperate deciduous  #
#      stand sheds its entire leaf pool every year, so annual leaf litterfall and the standing    #
#      leaf carbon should be the same number to within the evergreen fraction and the year\'s      #
#      growth. If they are orders apart, a unit is wrong somewhere -- which is exactly how the    #
#      v0.2 litterfall units defect was found. ---------------------------------------------------#
leaf_lit = np.asarray(m["litter_leaf_site"]).ravel()[SL].mean()
lc = read(M, ["leaf_carbon_site"])["leaf_carbon_site"]
print(f"leaf litterfall      {leaf_lit:7.3f} {units(M,'litter_leaf_site')}")
if lc is not None:
    print(f"standing leaf carbon {np.asarray(lc).ravel().mean():7.3f} {units(M,'leaf_carbon_site')}"
          f"   -> turnover {leaf_lit/max(np.asarray(lc).ravel().mean(),1e-9):.2f} /yr")

## 2. The diurnal cycle — the FAST tier

The fast tier writes one record per sub-step, one file per day. That is the only tier that can show
what the canopy does *within* a day, and the shape of that is the strongest free check on the
surface energy balance: net radiation peaking at solar noon, sensible and latent heat following it,
and the canopy air warmer than the free atmosphere by day.

A **mean diurnal composite** over a month, rather than one day, so weather does not dominate.

In [ ]:
def diurnal(year, month, names):
    """Mean diurnal composite of the named FAST variables over one calendar month."""
    files = [p for p in F if os.path.basename(p).split("-F-")[1][:6] == f"{year}{month:02d}"]
    if not files:
        return None, None
    v = read(files, names + ["hour", "minute"])
    hr = np.asarray(v["hour"]).ravel() + np.asarray(v["minute"]).ravel() / 60.0
    bins = np.arange(0, 25, 1.0)
    idx = np.digitize(hr, bins) - 1
    out = {}
    for n in names:
        if v[n] is None:
            out[n] = None ; continue
        a = np.asarray(v[n]).ravel()
        out[n] = np.array([a[idx == b].mean() if (idx == b).any() else np.nan
                           for b in range(24)])
    return bins[:-1] + 0.5, out

NAMES = ["rnet_fast", "h_flux_fast", "le_flux_fast", "sw_in_fast",
         "gpp_rate_fast", "nee_fast", "cas_temp_fast", "air_temp_fast"]
hrs, jul = diurnal(2075, 7, NAMES)
_,   jan = diurnal(2075, 1, NAMES)

if hrs is None:
    print("no FAST files for the requested month -- skipping the diurnal section")
else:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    for comp, name, ls in ((jul, "July", "-"), (jan, "January", "--")):
        if comp is None: continue
        ax[0].plot(hrs, comp["rnet_fast"], ls, color="k",         label=f"Rnet {name}")
        ax[0].plot(hrs, comp["h_flux_fast"], ls, color="firebrick",  label=f"H {name}")
        ax[0].plot(hrs, comp["le_flux_fast"], ls, color="royalblue", label=f"LE {name}")
        ax[1].plot(hrs, comp["gpp_rate_fast"], ls, color="seagreen", label=f"GPP {name}")
        if comp["nee_fast"] is not None:
            ax[1].plot(hrs, comp["nee_fast"], ls, color="sienna",   label=f"NEE {name}")
        ax[2].plot(hrs, comp["cas_temp_fast"] - 273.15, ls, color="darkorange", label=f"canopy air {name}")
        ax[2].plot(hrs, comp["air_temp_fast"] - 273.15, ls, color="0.5",        label=f"atmosphere {name}")
    ax[0].set_ylabel("flux [W/m2]") ; ax[0].set_title("surface energy balance")
    ax[1].set_ylabel("umol CO2/m2/s") ; ax[1].set_title("carbon exchange (NEE + to atmosphere)")
    ax[2].set_ylabel("temperature [degC]") ; ax[2].set_title("canopy air vs free atmosphere")
    for a in ax:
        a.set_xlabel("hour of day") ; a.set_xlim(0, 24) ; a.grid(alpha=0.3)
        a.axhline(0, color="k", lw=0.6) ; a.legend(fontsize=7)
    fig.tight_layout()
    plt.show()
    print(f"July midday Rnet  {np.nanmax(jul['rnet_fast']):7.1f} W/m2"
          f"   peak GPP {np.nanmax(jul['gpp_rate_fast']):6.2f} umol/m2/s")
    print(f"Jan  midday Rnet  {np.nanmax(jan['rnet_fast']):7.1f} W/m2"
          f"   peak GPP {np.nanmax(jan['gpp_rate_fast']):6.2f} umol/m2/s")

## 3. The seasonal cycle — the DAILY tier

Phenology, the carbon uptake it gates, and the water that leaves with it. Two things to read:

1. **Does the canopy have a season at all?** Check the LAI amplitude against the PFT's declared leaf
   habit *before* reading anything else. A PFT declared cold-deciduous whose LAI is flat is not a
   miscalibrated season — it is no season, and something upstream of the phenology is wrong.
2. **Are the three in phase?** Leaf area leads, GPP follows it, evapotranspiration follows GPP
   through the stomata. A GPP season running ahead of the canopy, or an ET season that does not
   track GPP, is a broken coupling rather than a tuning problem.

The first check is not hypothetical. It is what this notebook found on its first run: the Ithaca
reference stand, whose single PFT is declared `evergreen = [0]` with a temperature cue, held
LAI 5.47–5.66 through every January of a 50-year run. The cue was never reaching the model
(**#245**, fixed in v0.2.0 — the `[phenology]` block was silently skipped unless it carried a key
the shipped documentation never mentioned).

A run produced **before** that fix, or one whose PFT file still lacks a `[phenology]` section, is
evergreen regardless of what it declares. So check the amplitude first, every time.

In [ ]:
d = read(D, ["time", "year", "lai_site", "gpp_site", "npp_site", "et_site", "rh_site",
             "soil_wetness_site", "soil_temp_top_site", "precip_site", "swe_site"])
td = np.asarray(d["time"]).ravel()
fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

ax[0].plot(td, np.asarray(d["lai_site"]).ravel(), color="seagreen", lw=1.4, label="LAI")
ax[0].set_ylabel("LAI [m2/m2]", color="seagreen")
a0 = ax[0].twinx()
a0.plot(td, np.asarray(d["gpp_site"]).ravel(), color="darkgreen", lw=1.0, alpha=0.8, label="GPP")
if d["npp_site"] is not None:
    a0.plot(td, np.asarray(d["npp_site"]).ravel(), color="olive", lw=1.0, alpha=0.8, label="NPP")
a0.set_ylabel(f"carbon flux [{units(D,'gpp_site')}]")
a0.legend(fontsize=8, loc="upper right")
ax[0].set_title("canopy and carbon uptake")

if d["et_site"] is not None:
    ax[1].plot(td, np.asarray(d["et_site"]).ravel(), color="royalblue", lw=1.0, label="ET")
if d["precip_site"] is not None:
    ax[1].plot(td, np.asarray(d["precip_site"]).ravel(), color="0.6", lw=0.8, label="precipitation")
ax[1].set_ylabel(f"water [{units(D,'et_site')}]") ; ax[1].legend(fontsize=8)
ax[1].set_title("water: what arrives and what leaves")

#----- soil_wetness_site keeps the SOIL axis: `_site` means aggregated over PATCHES, not reduced  #
#      to a scalar. Take the top layer and label it with its own depth from the file\'s soil_z     #
#      coordinate rather than assuming a grid. ---------------------------------------------------#
wet = np.asarray(d["soil_wetness_site"])
with Dataset(D[0]) as _d:
    z_top = float(_d.variables["soil_z"][0]) if "soil_z" in _d.variables else float("nan")
ax[2].plot(td, wet[:, 0], color="saddlebrown", lw=1.2, label="soil wetness (top layer)")
ax[2].set_ylabel(f"soil wetness at z={z_top:.2f} m [-]", color="saddlebrown")
a2 = ax[2].twinx()
a2.plot(td, np.asarray(d["soil_temp_top_site"]).ravel() - 273.15, color="firebrick", lw=1.0,
        label="soil T (top)")
if d["swe_site"] is not None:
    a2.plot(td, np.asarray(d["swe_site"]).ravel(), color="steelblue", lw=1.0, label="snow SWE")
a2.set_ylabel("soil temp [degC] / SWE [kg/m2]") ; a2.legend(fontsize=8, loc="upper right")
ax[2].set_title("the ground") ; ax[2].set_xlabel("year")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout() ; plt.show()

lai = np.asarray(d["lai_site"]).ravel() ; gpp = np.asarray(d["gpp_site"]).ravel()

#----- Phase is read off the LAST COMPLETE CALENDAR YEAR, not the whole series. A run restarted  #
#      from a state file written under different phenology spends its first weeks relaxing out    #
#      of that state -- here the restart carried a full evergreen canopy into January, so LAI's    #
#      series maximum is record 0 and the whole-series "lag" is an artefact of the spin-out, not   #
#      a phase relationship. Use a year the model actually generated.  ---------------------------#
yr = np.asarray(d["year"]).ravel() if d["year"] is not None else None
if yr is not None and (yr == yr.max()).sum() >= 300:
    sel = yr == yr.max()
elif yr is not None and (yr == yr.max() - 1).sum() >= 300:
    sel = yr == yr.max() - 1
else:
    sel = np.zeros_like(lai, dtype=bool) ; sel[-min(365, lai.size):] = True
lai_y, gpp_y = lai[sel], gpp[sel]
label = f"{int(yr[sel][0])}" if yr is not None else "last ~365 records"

amp = (lai_y.max() - lai_y.min()) / max(lai_y.max(), 1e-9)
print(f"phase read off {label} ({sel.sum()} records)")
print(f"LAI  range {lai_y.min():.2f} .. {lai_y.max():.2f}   relative amplitude {amp:.1%}")
print(f"GPP  range {gpp_y.min():.3f} .. {gpp_y.max():.3f} {units(D,'gpp_site')}")
if amp < 0.25:
    print()
    print("   *** The canopy has essentially NO seasonal cycle.")
    print("   *** If any PFT in this run is declared deciduous, the phenology is not reaching the")
    print("   *** model. Before v0.2.0 that was true of EVERY run (#245); after it, check that the")
    print("   *** PFT file has a [phenology] section using flush_cue_mask / shed_cue_mask.")
else:
    #----- Compared on canopy ONSET, not on argmax. A flushed canopy plateaus for months and then  #
    #      creeps up a few percent with the season's growth, so its argmax lands wherever that      #
    #      creep happens to top out -- here day 262, against a canopy that was within 5% of full    #
    #      by day 150. Differencing two argmaxes on a plateau measures the creep, not the phase.    #
    #      Onset (first day at 95% of the year's maximum) is the date the canopy is actually there. #
    thresh = 0.95 * lai_y.max()
    onset  = int(np.argmax(lai_y >= thresh)) + 1
    gmax   = int(gpp_y.argmax()) + 1
    print(f"   canopy reaches 95% of its peak on day {onset}; GPP peaks on day {gmax}"
          f"  -> {gmax - onset:+d} days")
    print("   GPP should peak at or after the canopy is up, never long before it: the canopy is")
    print("   what does the fixing. A large negative number means carbon uptake is running ahead")
    print("   of the leaf area that is supposed to be producing it.")

## 4. The demographic trajectory — MONTHLY and ANNUAL

What separates a demographic model from a big-leaf one: the stand is a population, and its biomass
is an outcome of who is growing and who is dying rather than a state variable in its own right.

The **mortality split by pathway** is the v0.2 addition that makes this readable. Background
thinning, culls at the tracking floor, and canopy killed by treefall disturbance remove the same
carbon and mean entirely different things.

In [ ]:
names = ["time", "agb_site", "nplant_site", "basal_area_site", "mean_dbh_site",
         "n_cohort_site", "n_patch_site", "mort_carbon_background_site",
         "mort_carbon_cull_site", "mort_carbon_disturb_site", "agb_growth_site",
         "nplant_recruit_site", "disturb_area_site"]
m = read(M, names)
t = np.asarray(m["time"]).ravel()

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
ax[0,0].plot(t, np.asarray(m["agb_site"]).ravel(), color="darkgreen", lw=1.6)
ax[0,0].set_ylabel(f"AGB [{units(M,'agb_site')}]") ; ax[0,0].set_title("aboveground biomass")

a = ax[0,1]
a.plot(t, np.asarray(m["nplant_site"]).ravel(), color="steelblue", lw=1.4, label="stem density")
a.set_ylabel(f"stems [{units(M,'nplant_site')}]", color="steelblue")
a2 = a.twinx()
a2.plot(t, np.asarray(m["mean_dbh_site"]).ravel(), color="sienna", lw=1.4, label="mean DBH")
a2.set_ylabel("mean DBH [cm]", color="sienna")
a.set_title("self-thinning: fewer, bigger stems")

#----- Mortality by pathway (#169). The three partition whole-individual mortality carbon. -------#
a = ax[1,0]
paths = [("mort_carbon_background_site", "background hazard", "steelblue"),
         ("mort_carbon_cull_site",       "cull (size floor)", "goldenrod"),
         ("mort_carbon_disturb_site",    "treefall disturbance", "firebrick")]
have = [(k, lab, c) for k, lab, c in paths if m[k] is not None]
if have:
    a.stackplot(t, *[np.asarray(m[k]).ravel() for k, _, _ in have],
                labels=[lab for _, lab, _ in have], colors=[c for _, _, c in have], alpha=0.85)
    a.legend(fontsize=8, loc="upper left")
    SLm, Wm, LOm, HIm = windows(t)
    span = t[HIm] - t[LOm]
    amt = {k: (np.asarray(m[k]).ravel()[SLm] * Wm).sum() for k, _, _ in have}
    tot = sum(amt.values())
    print(f"mortality carbon by pathway, integrated over {span:.2f} yr [kgC/m2]:")
    for k, lab, _ in have:
        print(f"   {lab:<24} {amt[k]:7.4f}   ({amt[k]/max(tot,1e-12):5.1%})"
              f"   = {amt[k]/span:6.4f} kgC/m2/yr")
    print(f"   {'TOTAL':<24} {tot:7.4f}            = {tot/span:6.4f} kgC/m2/yr")
    print()
    print("   INTEGRATED with each window's own width, not averaged over records: a once-a-year")
    print("   event landing in a short final window reports a huge annualized rate, correctly, and")
    print("   a naive mean over records would let that one number dominate the whole split.")
    print("\n   A hard zero in the cull row is normally PHYSICAL, not a missing writer: at the\n"
          "   shipped negligible_nplant = 1e-8 no cohort reaches the tracking floor, because\n"
          "   cohort fusion consolidates small cohorts long before they decay that far.")
a.set_ylabel(f"[{units(M,'mort_carbon_background_site')}]") ; a.set_title("mortality carbon by pathway")

a = ax[1,1]
a.plot(t, np.asarray(m["agb_growth_site"]).ravel(), color="seagreen", lw=1.3, label="AGB growth")
tot_mort = sum(np.asarray(m[k]).ravel() for k, _, _ in have) if have else None
if tot_mort is not None:
    a.plot(t, tot_mort, color="firebrick", lw=1.3, label="mortality (all pathways)")
a.set_ylabel(f"[{units(M,'agb_growth_site')}]") ; a.legend(fontsize=8)
a.set_title("growth against mortality -- the net is the trajectory above")

for row in ax:
    for a in row: a.grid(alpha=0.3) ; a.set_xlabel("year")
fig.tight_layout() ; plt.show()

In [ ]:
#----- The size structure, from the PFT/size plotter's own axes. post_proc/plot_pft_size.py draws #
#      this as a standalone figure and prints the closure identities; here is the one number that  #
#      matters most -- whether the stand is moving carbon into larger trees. ---------------------#
if Y:
    with Dataset(Y[0]) as ds:
        lo, hi = ds.variables["dbh_lower"][:], ds.variables["dbh_upper"][:]
        agb_size = ds.variables["agb_size"][:]
        yr = ds.variables["time"][:]
    labels = [f"{a:g}-{b:g}" for a, b in zip(lo, hi)]
    first, last = np.asarray(agb_size[0]), np.asarray(agb_size[-1])
    print(f"aboveground biomass by DBH class, {yr[0]:.0f} -> {yr[-1]:.0f}  [kgC/m2]")
    for lab, f0, f1 in zip(labels, first, last):
        bar = "#" * int(round(f1 * 4))
        print(f"   {lab:>8} cm   {f0:7.3f} -> {f1:7.3f}   {f1-f0:+7.3f}   {bar}")
    print(f"   {'total':>8}      {first.sum():7.3f} -> {last.sum():7.3f}   {last.sum()-first.sum():+7.3f}")
else:
    print("no annual file -- enable [output.annual] to see the size structure")

## 5. What this does and does not establish

**Established, if the numbers above came out clean:** the output is internally consistent. The
reduction axes partition the stand, the soil-carbon budget closes against the litter and
respiration the model reports, the surface energy balance closes to its storage term, and the
diurnal cycles have the right shape.

**Not established:** that any of it is *right*. Nothing here has been compared against an
observation. A model can be perfectly conservative and perfectly seasonal and still have the wrong
productivity, the wrong turnover and the wrong successional trajectory.

### What this pass actually found

Written against a three-year Ithaca run, the first execution of this notebook turned up three
defects. **All three were fixed in the same release**, which is the argument for having it:

| | |
|---|---|
| **#245** | The phenology cue never reached the model. Every PFT ran evergreen whatever it declared, because the config block was skipped unless a key the shipped documentation never mentioned was present. No MEDS run before v0.2.0 had a leaf-area cycle. If section 3 below shows a flat LAI on a stand you declared deciduous, check that your `[phenology]` block uses `flush_cue_mask` / `shed_cue_mask` — the old `cue_mask` is now rejected by name. |
| **#247** | Correcting that crashed the run: the cohort diagnostic block kept a stale `n` after a cull, and the output path wrote past the end of a buffer sized from the live count. Memory corruption, and not confined to phenology. |
| **#246** | The soil axis was written at its compile-time maximum with `0` / `NaN` padding instead of `_FillValue`, so reducing over it gave a 136 K soil column. The padding is now fill, and **masked by any CF-aware reader** — which is why section 3 takes the top layer explicitly rather than trusting the axis length. |

One documented-behaviour surprise survives, in section 1: `litter_*_site` does not carry all the
carbon entering the soil.

### Two cautions this release carries

- **`dt_fast = 900 s` is a spin-up setting.** It biases GPP roughly −33% and ET roughly −24%
  against a resolved reference, because leaf gas exchange is frozen across the step. Any absolute
  flux read off a 900 s run is a lower bound. See `docs/science/numerical_scheme.md`.
- **Nothing here has been scored against the Ithaca flux record**, which is the evaluation this
  repository still does not contain. That is the right next step.